# Gradio 和 Streamlit 教程

Gradio 和 Streamlit 都能把 Python 函数、数据分析流程和模型推理包装成浏览器里的交互应用。Gradio 更适合快速生成模型 Demo、输入输出组件和分享链接；Streamlit 更适合数据看板、内部工具和脚本式应用。


## 安装和启动方式

这两个框架不是当前项目的核心依赖。需要运行示例时，在项目根目录安装：

```bash
uv add gradio streamlit
```

Gradio 通常可以直接运行 Python 脚本或 notebook 代码单元。Streamlit 需要把代码保存为 `.py` 文件，然后用命令启动：

```bash
streamlit run app.py
```


## 怎么选

| 场景 | 优先选择 | 原因 |
| --- | --- | --- |
| 给模型函数做一个可分享 Demo | Gradio | `Interface` 能直接绑定函数、输入组件和输出组件 |
| 需要多个输入区域、事件流和布局 | Gradio | `Blocks` 更适合组合复杂交互 |
| 数据分析报告、指标看板、内部运营工具 | Streamlit | 脚本从上到下渲染，写法接近普通数据分析脚本 |
| 需要缓存数据读取或模型加载 | Streamlit | `st.cache_data` 和 `st.cache_resource` 适合分层缓存 |
| 需要多人协作迭代产品级前端 | 传统前端框架 | 这两个工具更偏快速原型和中后台工具 |


## 先写可测试的业务函数

无论用哪个框架，都应该先把业务逻辑写成普通 Python 函数。UI 只负责收集输入、调用函数、展示结果。


In [ ]:
from dataclasses import dataclass


@dataclass(frozen=True)
class ReviewResult:
    """文本审核结果。"""

    label: str
    score: float
    reason: str



POSITIVE_WORDS = {"good", "great", "excellent", "amazing", "喜欢", "推荐", "满意"}
NEGATIVE_WORDS = {"bad", "poor", "terrible", "awful", "失望", "糟糕", "差"}


def analyze_review(text: str) -> ReviewResult:
    """根据关键词给一段评价文本打一个演示用分数。"""
    normalized = text.strip().lower()
    if not normalized:
        return ReviewResult(label="empty", score=0.0, reason="请输入一段评价文本。")

    positive_hits = sum(word in normalized for word in POSITIVE_WORDS)
    negative_hits = sum(word in normalized for word in NEGATIVE_WORDS)
    raw_score = positive_hits - negative_hits

    if raw_score > 0:
        label = "positive"
    elif raw_score < 0:
        label = "negative"
    else:
        label = "neutral"

    score = min(1.0, max(0.0, 0.5 + raw_score * 0.25))
    reason = f"positive={positive_hits}, negative={negative_hits}"
    return ReviewResult(label=label, score=score, reason=reason)


analyze_review("这个工具很好用，我很满意")


ReviewResult(label='positive', score=0.75, reason='positive=1, negative=0')

## Gradio: 用 `Interface` 快速包装函数

`gr.Interface` 适合一个函数对应一组输入输出的场景。把函数、输入组件、输出组件和示例传进去，就能得到一个浏览器应用。


In [4]:
def build_gradio_interface_code() -> str:
    """返回一个可保存为 gradio_app.py 的最小 Gradio 应用。"""
    return '''
import gradio as gr


def analyze_review(text: str) -> dict[str, str | float]:
    normalized = text.strip().lower()
    positive_hits = sum(word in normalized for word in {"good", "great", "喜欢", "推荐", "满意"})
    negative_hits = sum(word in normalized for word in {"bad", "poor", "失望", "糟糕", "差"})
    raw_score = positive_hits - negative_hits
    label = "positive" if raw_score > 0 else "negative" if raw_score < 0 else "neutral"
    score = min(1.0, max(0.0, 0.5 + raw_score * 0.25))
    return {"label": label, "score": score, "reason": f"positive={positive_hits}, negative={negative_hits}"}


demo = gr.Interface(
    fn=analyze_review,
    inputs=gr.Textbox(lines=4, label="Review"),
    outputs=gr.JSON(label="Result"),
    examples=["这个工具很好用，我很满意", "The experience was poor and bad"],
    title="Review Analyzer",
)


if __name__ == "__main__":
    demo.launch()
'''.strip()


print(build_gradio_interface_code())


import gradio as gr


def analyze_review(text: str) -> dict[str, str | float]:
    normalized = text.strip().lower()
    positive_hits = sum(word in normalized for word in {"good", "great", "喜欢", "推荐", "满意"})
    negative_hits = sum(word in normalized for word in {"bad", "poor", "失望", "糟糕", "差"})
    raw_score = positive_hits - negative_hits
    label = "positive" if raw_score > 0 else "negative" if raw_score < 0 else "neutral"
    score = min(1.0, max(0.0, 0.5 + raw_score * 0.25))
    return {"label": label, "score": score, "reason": f"positive={positive_hits}, negative={negative_hits}"}


demo = gr.Interface(
    fn=analyze_review,
    inputs=gr.Textbox(lines=4, label="Review"),
    outputs=gr.JSON(label="Result"),
    examples=["这个工具很好用，我很满意", "The experience was poor and bad"],
    title="Review Analyzer",
)


if __name__ == "__main__":
    demo.launch()


## Gradio: 用 `Blocks` 组合布局和事件

`Blocks` 适合多个组件、多个按钮、分栏布局和事件回调。常见结构是：创建组件，定义事件，把按钮点击或输入变化绑定到函数。


In [5]:
def build_gradio_blocks_code() -> str:
    """返回一个带布局和事件的 Gradio Blocks 示例。"""
    return '''
import gradio as gr


def summarize(text: str, max_words: int) -> str:
    words = text.split()
    if not words:
        return "请输入需要摘要的文本。"
    summary = " ".join(words[:max_words])
    suffix = "..." if len(words) > max_words else ""
    return f"{summary}{suffix}"


with gr.Blocks(title="Text Helper") as demo:
    gr.Markdown("# Text Helper")
    with gr.Row():
        text = gr.Textbox(lines=8, label="Input")
        output = gr.Textbox(lines=8, label="Summary")
    max_words = gr.Slider(5, 50, value=20, step=1, label="Max words")
    submit = gr.Button("Summarize", variant="primary")

    submit.click(fn=summarize, inputs=[text, max_words], outputs=output)


if __name__ == "__main__":
    demo.launch()
'''.strip()


print(build_gradio_blocks_code())


import gradio as gr


def summarize(text: str, max_words: int) -> str:
    words = text.split()
    if not words:
        return "请输入需要摘要的文本。"
    summary = " ".join(words[:max_words])
    suffix = "..." if len(words) > max_words else ""
    return f"{summary}{suffix}"


with gr.Blocks(title="Text Helper") as demo:
    gr.Markdown("# Text Helper")
    with gr.Row():
        text = gr.Textbox(lines=8, label="Input")
        output = gr.Textbox(lines=8, label="Summary")
    max_words = gr.Slider(5, 50, value=20, step=1, label="Max words")
    submit = gr.Button("Summarize", variant="primary")

    submit.click(fn=summarize, inputs=[text, max_words], outputs=output)


if __name__ == "__main__":
    demo.launch()


## Streamlit: 脚本式页面

Streamlit 应用就是一个从上到下执行的 Python 脚本。用户改动控件时，脚本会重新运行，因此耗时的数据加载和模型初始化要做缓存。


In [6]:
def build_streamlit_app_code() -> str:
    """返回一个可保存为 streamlit_app.py 的 Streamlit 应用。"""
    return '''
import pandas as pd
import streamlit as st


st.set_page_config(page_title="Sales Dashboard", layout="wide")
st.title("Sales Dashboard")


@st.cache_data
def load_sales() -> pd.DataFrame:
    return pd.DataFrame(
        [
            {"region": "East", "product": "A", "revenue": 1200},
            {"region": "East", "product": "B", "revenue": 900},
            {"region": "West", "product": "A", "revenue": 1600},
            {"region": "West", "product": "B", "revenue": 700},
        ]
    )


sales = load_sales()
regions = st.sidebar.multiselect("Region", sorted(sales["region"].unique()), default=sorted(sales["region"].unique()))
filtered = sales[sales["region"].isin(regions)]

total_revenue = int(filtered["revenue"].sum())
st.metric("Total revenue", f"${total_revenue:,}")

left, right = st.columns(2)
with left:
    st.subheader("Raw data")
    st.dataframe(filtered, use_container_width=True)

with right:
    st.subheader("Revenue by product")
    chart_data = filtered.groupby("product", as_index=False)["revenue"].sum()
    st.bar_chart(chart_data, x="product", y="revenue")
'''.strip()


print(build_streamlit_app_code())


import pandas as pd
import streamlit as st


st.set_page_config(page_title="Sales Dashboard", layout="wide")
st.title("Sales Dashboard")


@st.cache_data
def load_sales() -> pd.DataFrame:
    return pd.DataFrame(
        [
            {"region": "East", "product": "A", "revenue": 1200},
            {"region": "East", "product": "B", "revenue": 900},
            {"region": "West", "product": "A", "revenue": 1600},
            {"region": "West", "product": "B", "revenue": 700},
        ]
    )


sales = load_sales()
regions = st.sidebar.multiselect("Region", sorted(sales["region"].unique()), default=sorted(sales["region"].unique()))
filtered = sales[sales["region"].isin(regions)]

total_revenue = int(filtered["revenue"].sum())
st.metric("Total revenue", f"${total_revenue:,}")

left, right = st.columns(2)
with left:
    st.subheader("Raw data")
    st.dataframe(filtered, use_container_width=True)

with right:
    st.subheader("Revenue by product")
    chart_data = filtered.groupby("produc

## Streamlit 状态和缓存

`st.session_state` 保存当前浏览器会话里的状态，适合计数器、聊天历史、筛选条件等用户级状态。`st.cache_data` 适合缓存可序列化数据结果，`st.cache_resource` 适合缓存数据库连接、模型对象这类资源。


In [7]:
def build_streamlit_state_code() -> str:
    """返回一个展示 session_state 和 cache_resource 的 Streamlit 示例。"""
    return '''
from dataclasses import dataclass

import streamlit as st


@dataclass
class TinyModel:
    name: str

    def predict(self, text: str) -> str:
        return f"{self.name}: {len(text)} chars"


@st.cache_resource
def load_model() -> TinyModel:
    return TinyModel(name="demo-model")


model = load_model()
st.title("Session State Demo")

if "history" not in st.session_state:
    st.session_state.history = []

text = st.text_input("Text")
if st.button("Predict") and text:
    result = model.predict(text)
    st.session_state.history.append({"text": text, "result": result})

st.write(st.session_state.history)
'''.strip()


print(build_streamlit_state_code())


from dataclasses import dataclass

import streamlit as st


@dataclass
class TinyModel:
    name: str

    def predict(self, text: str) -> str:
        return f"{self.name}: {len(text)} chars"


@st.cache_resource
def load_model() -> TinyModel:
    return TinyModel(name="demo-model")


model = load_model()
st.title("Session State Demo")

if "history" not in st.session_state:
    st.session_state.history = []

text = st.text_input("Text")
if st.button("Predict") and text:
    result = model.predict(text)
    st.session_state.history.append({"text": text, "result": result})

st.write(st.session_state.history)


## 推荐项目结构

Demo 可以只有一个文件，但真实项目建议把业务逻辑和 UI 分开：

```text
my_app/
├── app.py              # Streamlit 或 Gradio 入口
├── services.py         # 数据读取、模型推理、外部 API
├── schemas.py          # dataclass / pydantic 模型
├── tests/
│   └── test_services.py
└── README.md
```

这样可以单独测试 `services.py`，UI 层只保留组件、布局和事件绑定。


## 常见注意点

- 不要在 UI 回调里硬编码密钥，使用环境变量或托管平台的 secrets 配置。
- 不要把耗时初始化放在 Streamlit 顶层反复执行，优先使用缓存装饰器。
- Gradio 回调函数应该返回和输出组件匹配的数据类型，例如字符串、字典、图片路径或数组。
- 文件上传、图片处理和模型推理要限制大小，必要时做超时和异常处理。
- 原型可直接用内置服务启动；需要团队长期使用时，再考虑认证、日志、监控和反向代理。
